# B3IT safety extension — LoRA training (Qwen2.5-0.5B-Instruct)

This is Experiment 2, separate from the system-prompt-only Experiment 1A. The reported follow-up trains a **LoRA adapter** on 256 harmful/refusal pairs and 768 benign/helpful pairs (1:3), then compares the base model with the adapter using the same frozen B3IT border prompts and the same held-out 50 harmful + 50 benign questions. B3IT and LT are calculated on the identical samples.

**Run order:** attach `b3it-safety-lora-source.zip` as a Kaggle input dataset; enable one T4 GPU; run cells in order. The run first installs the CUDA/vLLM stack, then you restart the kernel, then run the remaining cells. LoRA training uses the GPU; vLLM is launched only afterward in separate processes to avoid GPU memory overlap.

The harmful/refusal training source is `iNLP-Lab/multilingual-safety` (English); benign/helpful examples come from `AmazonScience/FalseReject`. Both cards specify CC BY-NC 4.0. Use these artifacts for non-commercial research and cite the dataset authors. The held-out evaluation prompts are excluded from the trainer. The initial sample size and one training epoch are a controlled pilot, not a claim of comprehensive safety training.


In [ ]:
from pathlib import Path
import zipfile

input_root = Path('/kaggle/input')
source_root = Path('/kaggle/working/b3it-safety-lora-source')
bundles = list(input_root.rglob('b3it-safety-lora-source.zip')) if input_root.exists() else []

if bundles:
    source_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as zf:
        zf.extractall(source_root)
    project = source_root / 'track-llm-apis'
else:
    # Kaggle may unpack archive contents when the dataset is created.
    runner_files = list(input_root.rglob('train_safety_lora.py')) if input_root.exists() else []
    runner_files = [p for p in runner_files if p.parent.name == 'reproduction' and p.parent.parent.name == 'track-llm-apis']
    if runner_files:
        project = runner_files[0].parent.parent
    else:
        print('Kaggle inputs currently contain:', list(input_root.iterdir()) if input_root.exists() else 'NO /kaggle/input directory')
        raise FileNotFoundError('Add the Kaggle Dataset containing b3it-safety-lora-source.zip (or its extracted track-llm-apis folder), then rerun this cell.')

reproduction = project / 'reproduction'
assert (reproduction / 'run_safety_prompt_1a_v2.py').is_file()
print('Source:', project)


## Install the Kaggle GPU runtime

Run this once in a fresh Kaggle session. It pins the same CUDA 12.8 PyTorch and vLLM 0.10.2 combination used by the earlier reproduction notebook, then adds the training packages. Installation can take several minutes.


In [ ]:
%pip install -q uv
!uv pip install --system --reinstall torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!uv pip install --system --reinstall vllm==0.10.2 --torch-backend=cu128 --extra-index-url https://wheels.vllm.ai/0.10.2/
%pip install --no-cache-dir --force-reinstall --no-deps 'huggingface_hub==0.36.2' 'transformers==4.56.1' 'tokenizers==0.22.1'
%pip install -q 'datasets==3.3.2' 'peft==0.17.1' 'accelerate==1.10.1'


## Restart the Kaggle kernel

After the install cell finishes, use **Run → Restart & clear cell outputs** (or Kaggle's restart option). Then rerun the source-extraction cell at the top to restore its variables, and continue with the diagnostic cell. Skip the installer after the restart unless packages are missing.


In [ ]:
import torch, transformers, tokenizers, huggingface_hub, vllm, datasets, peft, accelerate
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Transformers:', transformers.__version__, '| Tokenizers:', tokenizers.__version__)
print('Hugging Face Hub:', huggingface_hub.__version__, '| vLLM:', vllm.__version__)
print('datasets:', datasets.__version__, '| PEFT:', peft.__version__, '| GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, restart the session, then run this cell.')


In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path('/kaggle/working/models/Qwen2.5-0.5B-Instruct')
if not (model_dir / 'config.json').exists():
    snapshot_download(repo_id='Qwen/Qwen2.5-0.5B-Instruct', local_dir=str(model_dir))
print('Base model ready:', model_dir)


## Train the safety LoRA

This is LoRA fine-tuning, not full-model fine-tuning. The reported 1:3 follow-up uses 256 harmful/refusal + 768 benign/helpful rows, one epoch, per-device batch 1, gradient accumulation 8, rank 8, and learning rate 1e-4. The trainer masks system/user tokens from the loss so the model learns from assistant responses. It excludes every held-out V2 evaluation question. Training uses a GPU; expect the duration to vary with Kaggle's T4 availability and dataset download speed.


In [ ]:
import os, subprocess, sys
os.environ['B3IT_SAFETY_MODEL'] = str(model_dir)
os.environ['B3IT_LORA_DIR'] = '/kaggle/working/safety-lora-qwen05'
os.environ['B3IT_LORA_N_HARMFUL'] = '256'
os.environ['B3IT_LORA_N_BENIGN'] = '768'
subprocess.run([sys.executable, str(reproduction / 'train_safety_lora.py')], cwd=project, env=os.environ.copy(), check=True)


## Sample the unchanged base model

This creates neutral reference and neutral control observations, and evaluates the held-out questions on the base model. It reuses the same frozen calibration file that was used by the system-prompt experiment.


In [ ]:
import os, subprocess, sys
os.environ['B3IT_SAFETY_MODEL'] = str(model_dir)
os.environ['B3IT_LORA_MERGED_MODEL'] = '/kaggle/working/safety-lora-qwen05/merged_model'
os.environ['B3IT_LORA_RESULTS'] = '/kaggle/working/safety-lora-results'
subprocess.run([sys.executable, str(reproduction / 'run_safety_lora_eval.py'), '--phase', 'base'], cwd=project, env=os.environ.copy(), check=True)


## Sample the LoRA model

This loads the merged adapter model in a fresh process, samples the same border-input prompts under the same neutral system prompt, and answers the same held-out safety questions. The LoRA model is compared as a model change; no safety system prompt is added at inference time.


In [ ]:
subprocess.run([sys.executable, str(reproduction / 'run_safety_lora_eval.py'), '--phase', 'lora'], cwd=project, env=os.environ.copy(), check=True)


## Compute B3IT/LT comparison and create figures

The analysis uses the saved reference/control/treatment samples. Review all harmful and benign answers manually before interpreting safety behaviour; the refusal keyword flag is only a rough screening measure.


In [ ]:
subprocess.run([sys.executable, str(reproduction / 'analyze_safety_lora.py')], cwd=project, env=os.environ.copy(), check=True)
from IPython.display import Image, display
display(Image(filename='/kaggle/working/safety-lora-results/safety_lora_results.png'))


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from pathlib import Path

results_dir = Path('/kaggle/working/safety-lora-results')
artifact = Path('/kaggle/working/b3it-safety-lora-results.zip')
with ZipFile(artifact, 'w', ZIP_DEFLATED) as archive:
    for path in results_dir.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(results_dir.parent))
    manifest = Path('/kaggle/working/safety-lora-qwen05/training_manifest.json')
    if manifest.exists():
        archive.write(manifest, 'training_manifest.json')
print('Download artifact:', artifact)
print('Also review:', results_dir / 'safety_eval_for_review.csv')
